## **Desarrollo**

1. Se crea base de datos y tabla en Hive a la cual se le insertan datos
2. A esa tabla se llama desde Spark como Dataframe
3. A ese Dataframe se le reeemplaza los valores vacios como nulls al momento de crear la tabla Hive
4. Se utiliza un archivo de configuracion JSON, del cual se tomaran ciertas reglas para la transformacion del Dataframe
5. Y de acuerdo a esas reglas se reemplaza ese valor null, ya sea, por la palabra "Unknown" o 0
6. El Dataframe ya transformado se persiste en Postgres, en una nueva tabla Hive y en un archivo CSV

### **`Paso 1` - Crear base de datos y schema en Postgres**

In [ ]:
psql -U airflow --db postgres
CREATE database sparkdb;
CREATE SCHEMA sparkschema;

In [ ]:
C:\Users\Alfonso>docker exec -it postgres bash
bash-5.0# psql -U airflow --db postgres
psql (11.4)
Type "help" for help.

postgres=# CREATE database sparkdb;
CREATE DATABASE
postgres=#
postgres=# \c sparkdb
You are now connected to database "sparkdb" as user "airflow".
sparkdb=#
sparkdb=# CREATE SCHEMA sparkschema;
CREATE SCHEMA
sparkdb=#
sparkdb=# \l
                                List of databases
    Name    |  Owner  | Encoding |  Collate   |   Ctype    |  Access privileges
------------+---------+----------+------------+------------+---------------------
 airflow_db | airflow | UTF8     | en_US.utf8 | en_US.utf8 |
 metastore  | airflow | UTF8     | en_US.utf8 | en_US.utf8 | =Tc/airflow        +
            |         |          |            |            | airflow=CTc/airflow+
            |         |          |            |            | hive=CTc/airflow
 postgres   | airflow | UTF8     | en_US.utf8 | en_US.utf8 |
 sparkdb    | airflow | UTF8     | en_US.utf8 | en_US.utf8 |
 template0  | airflow | UTF8     | en_US.utf8 | en_US.utf8 | =c/airflow         +
            |         |          |            |            | airflow=CTc/airflow
 template1  | airflow | UTF8     | en_US.utf8 | en_US.utf8 | =c/airflow         +
            |         |          |            |            | airflow=CTc/airflow
(6 rows)

sparkdb=# \dn
    List of schemas
    Name     |  Owner
-------------+---------
 public      | airflow
 sparkschema | airflow
(2 rows)

sparkdb=#

CREATE TABLE IF NOT EXISTS sparkschema.spark_table
(
    id_curso integer NOT NULL,
    nombre_curso character varying COLLATE pg_catalog."default" NOT NULL,
    nombre_autor character varying COLLATE pg_catalog."default" NOT NULL,
    course_section json NOT NULL,
    creation_date date NOT NULL,
    CONSTRAINT spark_table_pkey PRIMARY KEY (course_id)
)

### **`Paso 2` - Crear directorio para base de datos en Hive**

In [ ]:
# Crear ruta de la base de datos
hdfs dfs -mkdir -p /user/hive/warehouse/sparkdb 

In [ ]:
root@spark-master:/# hdfs dfs -mkdir -p /user/hive/warehouse/sparkdb
root@spark-master:/# 
root@spark-master:/# hdfs dfs -ls /user/hive/warehouse
Found 1 items
drwxr-xr-x   - root supergroup          0 2024-07-23 01:50 /user/hive/warehouse/sparkdb
root@spark-master:/#

### **`Paso 3`**: **Despliegue**

In [ ]:
spark-submit --class Main.SparkScala scala-spark-1.0-SNAPSHOT.jar

### **`Paso 4`**: **Verificación de tablas en `HIVE`** 

In [ ]:
C:\Users\Alfonso>docker exec -it hive-server bash
root@hive-server:/opt# beeline -u jdbc:hive2://localhost:10000
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.10.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop/share/hadoop/common/lib/slf4j-log4j12-1.7.25.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://localhost:10000
Connected to: Apache Hive (version 3.1.2)
Driver: Hive JDBC (version 3.1.2)
Transaction isolation: TRANSACTION_REPEATABLE_READ
Beeline version 3.1.2 by Apache Hive
0: jdbc:hive2://localhost:10000> SHOW DATABASES;
DEBUG : Acquired the compile lock.
INFO  : Compiling command(queryId=root_20240724211434_21df59d7-eedf-4f12-9751-1959716ef18e): SHOW DATABASES
INFO  : Concurrency mode is disabled, not creating a lock manager
INFO  : Semantic Analysis Completed (retrial = false)
INFO  : Returning Hive schema: Schema(fieldSchemas:[FieldSchema(name:database_name, type:string, comment:from deserializer)], properties:null)
INFO  : Completed compiling command(queryId=root_20240724211434_21df59d7-eedf-4f12-9751-1959716ef18e); Time taken: 1.733 seconds
INFO  : Concurrency mode is disabled, not creating a lock manager
INFO  : Executing command(queryId=root_20240724211434_21df59d7-eedf-4f12-9751-1959716ef18e): SHOW DATABASES
INFO  : Starting task [Stage-0:DDL] in serial mode
INFO  : Completed executing command(queryId=root_20240724211434_21df59d7-eedf-4f12-9751-1959716ef18e); Time taken: 0.064 seconds
INFO  : OK
INFO  : Concurrency mode is disabled, not creating a lock manager
DEBUG : Shutting down query SHOW DATABASES
+----------------+
| database_name  |
+----------------+
| default        |
| sparkdb        |
+----------------+
2 rows selected (2.579 seconds)
0: jdbc:hive2://localhost:10000> SELECT * FROM sparkdb.tabla_cursos;
DEBUG : Acquired the compile lock.
INFO  : Compiling command(queryId=root_20240724211454_659fe3d4-65e7-4949-b0e2-2dbd49377de2): SELECT * FROM sparkdb.tabla_cursos
INFO  : Concurrency mode is disabled, not creating a lock manager
INFO  : Semantic Analysis Completed (retrial = false)
INFO  : Returning Hive schema: Schema(fieldSchemas:[FieldSchema(name:tabla_cursos.id_curso, type:string, comment:null), FieldSchema(name:tabla_cursos.nombre_curso, type:string, comment:null), FieldSchema(name:tabla_cursos.nombre_autor, type:string, comment:null), FieldSchema(name:tabla_cursos.num_opiniones, type:string, comment:null)], properties:null)
INFO  : Completed compiling command(queryId=root_20240724211454_659fe3d4-65e7-4949-b0e2-2dbd49377de2); Time taken: 3.229 seconds
INFO  : Concurrency mode is disabled, not creating a lock manager
INFO  : Executing command(queryId=root_20240724211454_659fe3d4-65e7-4949-b0e2-2dbd49377de2): SELECT * FROM sparkdb.tabla_cursos
INFO  : Completed executing command(queryId=root_20240724211454_659fe3d4-65e7-4949-b0e2-2dbd49377de2); Time taken: 0.002 seconds
INFO  : OK
INFO  : Concurrency mode is disabled, not creating a lock manager
DEBUG : Shutting down query SELECT * FROM sparkdb.tabla_cursos
+------------------------+----------------------------+----------------------------+-----------------------------+
| tabla_cursos.id_curso  | tabla_cursos.nombre_curso  | tabla_cursos.nombre_autor  | tabla_cursos.num_opiniones  |
+------------------------+----------------------------+----------------------------+-----------------------------+
| 4                      | Linux                      | Future                     | 100                         |
| 1                      | Java                       | FutureX                    | 45                          |
| 11                     | Jenkins                    | Future                     | 32                          |
| 6                      | CMS                        | NULL                       | 100                         |
| 12                     | Chef                       | FutureX                    | 121                         |
| 7                      | Python                     | FutureX                    | NULL                        |
| 5                      | Microservices              | Future                     | 100                         |
| 2                      | Java                       | FutureXSkill               | 56                          |
| 8                      | CMS                        | Future                     | 56                          |
| 10                     | Ansible                    | FutureX                    | 123                         |
| 13                     | Go Lang                    | NULL                       | 105                         |
| 3                      | Big Data                   | Future                     | 100                         |
| 9                      | Dot Net                    | FutureXSkill               | 34                          |
+------------------------+----------------------------+----------------------------+-----------------------------+
13 rows selected (3.88 seconds)
0: jdbc:hive2://localhost:10000> SELECT * FROM sparkdb.nueva_tabla_cursos;
DEBUG : Acquired the compile lock.
INFO  : Compiling command(queryId=root_20240724211504_a5d28fbd-a3d6-4c9d-be52-32045707282f): SELECT * FROM sparkdb.nueva_tabla_cursos
INFO  : Concurrency mode is disabled, not creating a lock manager
INFO  : Semantic Analysis Completed (retrial = false)
INFO  : Returning Hive schema: Schema(fieldSchemas:[FieldSchema(name:nueva_tabla_cursos.id_curso, type:string, comment:null), FieldSchema(name:nueva_tabla_cursos.nombre_curso, type:string, comment:null), FieldSchema(name:nueva_tabla_cursos.nombre_autor, type:string, comment:null), FieldSchema(name:nueva_tabla_cursos.num_opiniones, type:string, comment:null)], properties:null)
INFO  : Completed compiling command(queryId=root_20240724211504_a5d28fbd-a3d6-4c9d-be52-32045707282f); Time taken: 0.384 seconds
INFO  : Concurrency mode is disabled, not creating a lock manager
INFO  : Executing command(queryId=root_20240724211504_a5d28fbd-a3d6-4c9d-be52-32045707282f): SELECT * FROM sparkdb.nueva_tabla_cursos
INFO  : Completed executing command(queryId=root_20240724211504_a5d28fbd-a3d6-4c9d-be52-32045707282f); Time taken: 0.001 seconds
INFO  : OK
INFO  : Concurrency mode is disabled, not creating a lock manager
DEBUG : Shutting down query SELECT * FROM sparkdb.nueva_tabla_cursos
+------------------------------+----------------------------------+----------------------------------+-----------------------------------+
| nueva_tabla_cursos.id_curso  | nueva_tabla_cursos.nombre_curso  | nueva_tabla_cursos.nombre_autor  | nueva_tabla_cursos.num_opiniones  |
+------------------------------+----------------------------------+----------------------------------+-----------------------------------+
| 4                            | Linux                            | Future                           | 100                               |
| 1                            | Java                             | FutureX                          | 45                                |
| 11                           | Jenkins                          | Future                           | 32                                |
| 6                            | CMS                              | Unknown                          | 100                               |
| 12                           | Chef                             | FutureX                          | 121                               |
| 7                            | Python                           | FutureX                          | NULL                              |
| 5                            | Microservices                    | Future                           | 100                               |
| 2                            | Java                             | FutureXSkill                     | 56                                |
| 8                            | CMS                              | Future                           | 56                                |
| 10                           | Ansible                          | FutureX                          | 123                               |
| 13                           | Go Lang                          | Unknown                          | 105                               |
| 3                            | Big Data                         | Future                           | 100                               |
| 9                            | Dot Net                          | FutureXSkill                     | 34                                |
+------------------------------+----------------------------------+----------------------------------+-----------------------------------+
13 rows selected (0.614 seconds)
0: jdbc:hive2://localhost:10000>

### **`Paso 5`**: **Verificacion de tablas Hive y archivo output CSV en `HDFS`** 

In [ ]:
C:\Users\Alfonso>docker exec -it namenode bash
root@namenode:/#
root@namenode:/# hdfs dfs -ls /user/hive/warehouse
Found 1 items
drwxr-xr-x   - root supergroup          0 2024-07-24 21:11 /user/hive/warehouse/sparkdb
root@namenode:/#
root@namenode:/# hdfs dfs -ls /user/hive/warehouse/sparkdb
Found 2 items
drwxr-xr-x   - root supergroup          0 2024-07-24 21:11 /user/hive/warehouse/sparkdb/nueva_tabla_cursos
drwxr-xr-x   - root supergroup          0 2024-07-24 21:11 /user/hive/warehouse/sparkdb/tabla_cursos
root@namenode:/#
root@namenode:/# hdfs dfs -ls /user/hive/warehouse/sparkdb/nueva_tabla_cursos
Found 13 items
-rwxr-xr-x   3 root supergroup         19 2024-07-24 21:11 /user/hive/warehouse/sparkdb/nueva_tabla_cursos/part-00000-a9d69c0f-3fb7-407e-bf9e-99a3d34ac2d5-c000
-rwxr-xr-x   3 root supergroup         18 2024-07-24 21:11 /user/hive/warehouse/sparkdb/nueva_tabla_cursos/part-00001-a9d69c0f-3fb7-407e-bf9e-99a3d34ac2d5-c000
-rwxr-xr-x   3 root supergroup         21 2024-07-24 21:11 /user/hive/warehouse/sparkdb/nueva_tabla_cursos/part-00002-a9d69c0f-3fb7-407e-bf9e-99a3d34ac2d5-c000
-rwxr-xr-x   3 root supergroup         18 2024-07-24 21:11 /user/hive/warehouse/sparkdb/nueva_tabla_cursos/part-00003-a9d69c0f-3fb7-407e-bf9e-99a3d34ac2d5-c000
-rwxr-xr-x   3 root supergroup         20 2024-07-24 21:11 /user/hive/warehouse/sparkdb/nueva_tabla_cursos/part-00004-a9d69c0f-3fb7-407e-bf9e-99a3d34ac2d5-c000
-rwxr-xr-x   3 root supergroup         20 2024-07-24 21:11 /user/hive/warehouse/sparkdb/nueva_tabla_cursos/part-00005-a9d69c0f-3fb7-407e-bf9e-99a3d34ac2d5-c000
-rwxr-xr-x   3 root supergroup         27 2024-07-24 21:11 /user/hive/warehouse/sparkdb/nueva_tabla_cursos/part-00006-a9d69c0f-3fb7-407e-bf9e-99a3d34ac2d5-c000
-rwxr-xr-x   3 root supergroup         23 2024-07-24 21:11 /user/hive/warehouse/sparkdb/nueva_tabla_cursos/part-00007-a9d69c0f-3fb7-407e-bf9e-99a3d34ac2d5-c000
-rwxr-xr-x   3 root supergroup         16 2024-07-24 21:11 /user/hive/warehouse/sparkdb/nueva_tabla_cursos/part-00008-a9d69c0f-3fb7-407e-bf9e-99a3d34ac2d5-c000
-rwxr-xr-x   3 root supergroup         23 2024-07-24 21:11 /user/hive/warehouse/sparkdb/nueva_tabla_cursos/part-00009-a9d69c0f-3fb7-407e-bf9e-99a3d34ac2d5-c000
-rwxr-xr-x   3 root supergroup         23 2024-07-24 21:11 /user/hive/warehouse/sparkdb/nueva_tabla_cursos/part-00010-a9d69c0f-3fb7-407e-bf9e-99a3d34ac2d5-c000
-rwxr-xr-x   3 root supergroup         22 2024-07-24 21:11 /user/hive/warehouse/sparkdb/nueva_tabla_cursos/part-00011-a9d69c0f-3fb7-407e-bf9e-99a3d34ac2d5-c000
-rwxr-xr-x   3 root supergroup         26 2024-07-24 21:11 /user/hive/warehouse/sparkdb/nueva_tabla_cursos/part-00012-a9d69c0f-3fb7-407e-bf9e-99a3d34ac2d5-c000
root@namenode:/#
root@namenode:/# hdfs dfs -ls /user/hive/warehouse/sparkdb/tabla_cursos
Found 13 items
-rwxr-xr-x   3 root supergroup         19 2024-07-24 21:10 /user/hive/warehouse/sparkdb/tabla_cursos/part-00000-0953d833-d0fd-49fe-a997-fe08d101aa1a-c000
-rwxr-xr-x   3 root supergroup         18 2024-07-24 21:10 /user/hive/warehouse/sparkdb/tabla_cursos/part-00000-1a12780b-7957-4495-aaae-bd8f712ca70e-c000
-rwxr-xr-x   3 root supergroup         21 2024-07-24 21:10 /user/hive/warehouse/sparkdb/tabla_cursos/part-00000-20a26b89-38da-42fc-936f-b2c1f3c27a16-c000
-rwxr-xr-x   3 root supergroup         11 2024-07-24 21:10 /user/hive/warehouse/sparkdb/tabla_cursos/part-00000-2f55eb2a-aee0-4c8f-a542-ecc103476ba8-c000
-rwxr-xr-x   3 root supergroup         20 2024-07-24 21:10 /user/hive/warehouse/sparkdb/tabla_cursos/part-00000-30496963-543e-42a6-9d03-8cfa905081c7-c000
-rwxr-xr-x   3 root supergroup         18 2024-07-24 21:10 /user/hive/warehouse/sparkdb/tabla_cursos/part-00000-44eb65ea-5f04-4476-b990-43431f406905-c000
-rwxr-xr-x   3 root supergroup         27 2024-07-24 21:10 /user/hive/warehouse/sparkdb/tabla_cursos/part-00000-61526fba-878f-4273-9d2a-b77cbcf89165-c000
-rwxr-xr-x   3 root supergroup         23 2024-07-24 21:10 /user/hive/warehouse/sparkdb/tabla_cursos/part-00000-7aec09a5-cf6d-4fb5-8ea5-c47fe2f93fd4-c000
-rwxr-xr-x   3 root supergroup         16 2024-07-24 21:10 /user/hive/warehouse/sparkdb/tabla_cursos/part-00000-84b07dac-9413-4540-a770-a1e25f89421f-c000
-rwxr-xr-x   3 root supergroup         23 2024-07-24 21:10 /user/hive/warehouse/sparkdb/tabla_cursos/part-00000-890ef5a7-2064-4a72-82d9-899434defe4f-c000
-rwxr-xr-x   3 root supergroup         16 2024-07-24 21:10 /user/hive/warehouse/sparkdb/tabla_cursos/part-00000-8d13fee7-ab4d-4894-9c50-373acfdb0650-c000
-rwxr-xr-x   3 root supergroup         22 2024-07-24 21:10 /user/hive/warehouse/sparkdb/tabla_cursos/part-00000-ad63d400-7bc9-4168-9d51-aa43fe3a1eca-c000
-rwxr-xr-x   3 root supergroup         26 2024-07-24 21:10 /user/hive/warehouse/sparkdb/tabla_cursos/part-00000-ce648071-e54f-4276-99b9-c3c35388696c-c000
root@namenode:/#
root@namenode:/#
root@namenode:/# hdfs dfs -ls /user/hive/archivos
Found 14 items
-rw-r--r--   3 root supergroup          0 2024-07-24 22:39 /user/hive/archivos/_SUCCESS
-rw-r--r--   3 root supergroup         20 2024-07-24 22:39 /user/hive/archivos/part-00000-2f824376-be9b-469f-bdcb-9c5c1543fc37-c000.csv
-rw-r--r--   3 root supergroup         18 2024-07-24 22:39 /user/hive/archivos/part-00001-2f824376-be9b-469f-bdcb-9c5c1543fc37-c000.csv
-rw-r--r--   3 root supergroup         27 2024-07-24 22:39 /user/hive/archivos/part-00002-2f824376-be9b-469f-bdcb-9c5c1543fc37-c000.csv
-rw-r--r--   3 root supergroup         22 2024-07-24 22:39 /user/hive/archivos/part-00003-2f824376-be9b-469f-bdcb-9c5c1543fc37-c000.csv
-rw-r--r--   3 root supergroup         18 2024-07-24 22:39 /user/hive/archivos/part-00004-2f824376-be9b-469f-bdcb-9c5c1543fc37-c000.csv
-rw-r--r--   3 root supergroup         23 2024-07-24 22:39 /user/hive/archivos/part-00005-2f824376-be9b-469f-bdcb-9c5c1543fc37-c000.csv
-rw-r--r--   3 root supergroup         19 2024-07-24 22:39 /user/hive/archivos/part-00006-2f824376-be9b-469f-bdcb-9c5c1543fc37-c000.csv
-rw-r--r--   3 root supergroup         23 2024-07-24 22:39 /user/hive/archivos/part-00007-2f824376-be9b-469f-bdcb-9c5c1543fc37-c000.csv
-rw-r--r--   3 root supergroup         21 2024-07-24 22:39 /user/hive/archivos/part-00008-2f824376-be9b-469f-bdcb-9c5c1543fc37-c000.csv
-rw-r--r--   3 root supergroup         20 2024-07-24 22:39 /user/hive/archivos/part-00009-2f824376-be9b-469f-bdcb-9c5c1543fc37-c000.csv
-rw-r--r--   3 root supergroup         26 2024-07-24 22:39 /user/hive/archivos/part-00010-2f824376-be9b-469f-bdcb-9c5c1543fc37-c000.csv
-rw-r--r--   3 root supergroup         23 2024-07-24 22:39 /user/hive/archivos/part-00011-2f824376-be9b-469f-bdcb-9c5c1543fc37-c000.csv
-rw-r--r--   3 root supergroup         16 2024-07-24 22:39 /user/hive/archivos/part-00012-2f824376-be9b-469f-bdcb-9c5c1543fc37-c000.csv
root@namenode:/#

### **`Paso 6`**: **Verificacion de de tabla en `Postgres`** 

In [ ]:
C:\Users\Alfonso>docker exec -it postgres bash
bash-5.0#
bash-5.0# psql -U airflow --db postgres
psql (11.4)
Type "help" for help.

sparkdb=#
sparkdb=# \dn
    List of schemas
    Name     |  Owner
-------------+---------
 public      | airflow
 sparkschema | airflow
(2 rows)


sparkdb=# SELECT table_schema, table_name
sparkdb-# FROM information_schema.tables
sparkdb-# WHERE table_schema = 'sparkschema';
 table_schema |  table_name
--------------+--------------
 sparkschema  | tabla_cursos
(1 row)


sparkdb=# -- Consulta para ver los datos de la tabla tabla_cursos
sparkdb=# SELECT * FROM sparkschema.tabla_cursos;
 id_curso | nombre_curso  | nombre_autor | num_opiniones
----------+---------------+--------------+---------------
 1        | Java          | FutureX      | 45
 9        | Dot Net       | FutureXSkill | 34
 3        | Big Data      | Future       | 100
 7        | Python        | FutureX      |
 6        | CMS           | Unknown      | 100
 2        | Java          | FutureXSkill | 56
 5        | Microservices | Future       | 100
 8        | CMS           | Future       | 56
 11       | Jenkins       | Future       | 32
 13       | Go Lang       | Unknown      | 105
 12       | Chef          | FutureX      | 121
 10       | Ansible       | FutureX      | 123
 4        | Linux         | Future       | 100
(13 rows)

sparkdb=#